In [368]:
import pandas as pd
import numpy as np

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

pd.set_option("display.max_columns", 25)

NumPy: 1.26.4
Pandas: 3.0.5


## delete duplicate rows

In [369]:
df = pd.read_csv("../data/raw/store-data.csv")
# duplicate Row Id
df = df.drop_duplicates("Row ID")

## shipping mode

In [370]:
## change the data formate

df['Order Date'] = pd.to_datetime(
    df['Order Date'],
    format='mixed',
    errors='coerce'
)

df['Ship Date'] = pd.to_datetime(
    df['Ship Date'],
    format='mixed',
    errors='coerce'
)

In [371]:
# fill the ship mode and ship date using the same order id

df['Ship Mode'] = df.groupby('Order ID')['Ship Mode'].transform(lambda x : x.ffill().bfill())
df['Ship Date'] = df.groupby('Order ID')['Ship Date'].transform(lambda x : x.ffill().bfill())

# calcul estimated time for each shipping mode
# add column shipping days

df['Shipping Days'] = (
    df['Ship Date'] - df['Order Date']
).dt.days

# Calculate the typical shipping time for each Ship Mode

estimated_days = (df.groupby('Ship Mode')['Shipping Days'].median())

# Assign the estimated number of days based on Ship Mode
df['Estimated Days'] = df['Ship Mode'].map(estimated_days)

# Fill missing Ship Date

df['Ship Date'] = df['Ship Date'].fillna(
    df['Order Date'] + pd.to_timedelta(df['Estimated Days'], unit='D')
)

# Recalculate Shipping Days AFTER filling Ship Date
df['Shipping Days'] = (
    df['Ship Date'] - df['Order Date']
).dt.days

def estimate_ship_mode(days):
    if pd.isna(days):
        return pd.NA
    return (estimated_days - days).abs().idxmin()

df['Ship Mode'] = df['Ship Mode'].fillna(
    df['Shipping Days'].apply(estimate_ship_mode)
)


## Customer and Segment

In [372]:
# fill the Customer Name using the same Customer id

df['Customer Name'] = df.groupby('Customer ID')['Customer Name'].transform(lambda x: x.ffill().bfill())

# fill the ones who have no name
df['Customer Name'] = df['Customer Name'].fillna('Unknown')

# normalize names
df['Customer Name Normalized'] = (
    df['Customer Name']
    .str.lower()
    .str.split()
    .apply(lambda x: ' '.join(sorted(x)) if isinstance(x, list) else x)
)

segment_mapping = {
    'Home Ofice': 'Home Office',
    'Consumerr': 'Consumer',
    'Corporrate': 'Corporate'
}

df['Segment'] = df['Segment'].replace(segment_mapping)

## City, State, Postal Code and Region

In [373]:
# titled the cites
df['City'] = df['City'].str.strip().str.title()
# states was good
# post code
df['Postal Code'] = df.groupby('City')['Postal Code'].transform(lambda x: x.ffill().bfill())
# region was good


## Product, Category and sub_category

In [374]:
# correct the product name with the most frequent one

df['Product Name'] = (
    df.groupby('Product ID')['Product Name'].transform(lambda x: x.mode()[0] if not x.mode().empty else x)
)
# Category
# the category name
df['Category'] = df['Category'].str.strip().str.title()

# sub_category was good



## sales, Quantity, Discount and Profite

In [375]:
# sales = Price * (1 - Discount) * Quantity
# product_price = sales / ( (1 - Discount) * Quantity )
# also we need product cost


df['Price'] = np.where(
    df['Sales'].notna() &
    df['Discount'].notna() &
    df['Quantity'].notna() &
    ((1 - df['Discount']) != 0),
    df['Sales'] / ((1 - df['Discount']) * df['Quantity']),
    np.nan
)
# fix the prices
df['Price'] = df.groupby('Product ID')['Price'].transform(lambda x: x.mode()[0] if not x.mode().empty else x)

# fix the sales
df['Sales'] = np.where(
    df['Sales'].isna() &
    df['Price'].notna() &
    df['Discount'].notna() &
    df['Quantity'].notna(),
    df['Price'] * (1 - df['Discount']) * df['Quantity'],
    df['Sales']
)

# fix the quantity
df['Quantity'] = np.where(
    df['Quantity'].isna() &
    df['Sales'].notna() &
    df['Discount'].notna() &
    df['Price'].notna() &
    ((1 - df['Discount']) != 0) &
    (df['Price'] != 0),
    df['Sales'] / ((1 - df['Discount']) * df['Price']),
    df['Quantity']
)

# we will fix the quantity and sales trough profit
# fixing the profit
# profit = sales - (product cost * Quantity)
# product cost = (sales - profit) / Quantity

df['Product Cost'] = np.where(
    df['Profit'].notna() &
    df['Quantity'].notna() &
    (df['Quantity'] != 0),

    (df['Sales'] - df['Profit']) / df['Quantity'],

    np.nan
)

# fix the Product cost
df['Product Cost'] = df.groupby('Product ID')['Product Cost'].transform(lambda x: x.mode()[0] if not x.mode().empty else x)

# fix the profit

df['Profit'] = np.where(
    df['Sales'].notna() &
    df['Product Cost'].notna() &
    df['Quantity'].notna(),

    df['Sales'] - (df['Product Cost'] * df['Quantity']),

    df['Profit']
)



# df[['Product ID', 'Product Name','Pro
df[df[['Sales', 'Price', 'Quantity']].isna().any(axis=1)][
    ['Product ID', 'Profit', 'Product Cost', 'Discount', 'Sales', 'Price', 'Quantity']
]

,Product ID,Profit,Product Cost,Discount,Sales,Price,Quantity
345,TEC-PH-10002293,4.7976,14.3928,0.2,NaN,19.99,NaN
347,OFF-PA-10000249,11.5432,6.5084,0.0,NaN,12.28,NaN
576,OFF-PA-10001450,9.7608,NaN,0.0,19.920,NaN,NaN
2877,OFF-EN-10003040,24.5998,13.8701,0.0,NaN,26.17,NaN
4100,OFF-AP-10004655,-5.2072,NaN,0.8,2.264,NaN,NaN
5894,OFF-ST-10003123,23.9688,25.3004,0.0,NaN,33.29,NaN
6574,TEC-MA-10004255,118.3704,NaN,0.0,NaN,NaN,8.0
6938,TEC-AC-10004420,52.7692,NaN,0.0,NaN,NaN,4.0
7755,OFF-PA-10002558,93.8840,NaN,0.2,NaN,NaN,7.0
8336,OFF-AR-10003986,3.1570,NaN,0.0,NaN,NaN,2.0
